In [10]:
import os
import pandas as pd

PROJECT_ROOT = os.path.abspath("..")  # from notebooks → src
DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "raw",
    "house-prices-advanced-regression-techniques",
    "train.csv"
)

df = pd.read_csv(DATA_PATH)

print("Loaded shape:", df.shape)

Loaded shape: (1460, 81)


In [13]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

from features.build_features import build_features
from features.feature_selector import select_features

# ======================================================
# 1️⃣ Load Data (robust path)
# ======================================================

PROJECT_ROOT = os.path.abspath("..")  # notebook inside src/notebooks
DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "raw",
    "house-prices-advanced-regression-techniques",
    "train.csv"
)

df = pd.read_csv(DATA_PATH)

print("Original shape:", df.shape)

# ======================================================
# 2️⃣ Feature Engineering
# ======================================================

df = build_features(df)

print("After feature engineering:", df.shape)

# ======================================================
# 3️⃣ Train/Test Split (Regression → NO stratify)
# ======================================================

X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape)

# ======================================================
# 4️⃣ Keep Only Numeric Features (for selection stage)
# ======================================================

X_train_numeric = X_train.select_dtypes(include=["int64", "float64"])

print("Numeric features:", X_train_numeric.shape[1])

# ======================================================
# 5️⃣ Impute Missing Values (Median → robust)
# ======================================================

imputer = SimpleImputer(strategy="median")

X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train_numeric),
    columns=X_train_numeric.columns,
    index=X_train_numeric.index
)

print("Any NaNs remaining?",
      X_train_imputed.isna().sum().sum())

# ======================================================
# 6️⃣ Feature Selection
# ======================================================

FEATURE_JSON_PATH = os.path.join(
    PROJECT_ROOT,
    "features",
    "feature_list.json"
)

selected_features, mi_scores = select_features(
    X_train_imputed,
    y_train,
    save_path=FEATURE_JSON_PATH
)

print("Selected feature count:", len(selected_features))
print("First 10 selected features:", selected_features[:10])
print("JSON saved at:", FEATURE_JSON_PATH)

Original shape: (1460, 81)
After feature engineering: (1460, 91)
Train shape: (1168, 90)
Numeric features: 47
Any NaNs remaining? 0
Selected feature count: 28
First 10 selected features: ['MSSubClass', '1stFlrSF', 'LotArea', 'BsmtUnfSF', 'ScreenPorch', 'MasVnrArea', 'GrLivArea', 'OverallQual', 'LotFrontage', 'QualityIndex']
JSON saved at: /home/anshulgarg/Documents/ObsidianVault/Hestabit/Week6/day-2/src/features/feature_list.json


In [14]:
print("Max MI score:", mi_scores.max())
print("Min MI score:", mi_scores.min())
print("Are all selected in original?",
      all(f in X_train_numeric.columns for f in selected_features))

Max MI score: 0.6370912646042242
Min MI score: 0.0
Are all selected in original? True


In [15]:
import numpy as np

corr_matrix = X_train_numeric[selected_features].corr().abs()
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

print("Any correlation > 0.9 in final set?",
      (upper > 0.9).sum().sum() > 0)

Any correlation > 0.9 in final set? False


In [16]:
import json

json_path = os.path.join(PROJECT_ROOT, "features", "feature_list.json")

print("File exists:", os.path.exists(json_path))

with open(json_path, "r") as f:
    data = json.load(f)

print("Keys:", data.keys())
print("Final selected count:", len(data["final_selected"]))

File exists: True
Keys: dict_keys(['all_features', 'correlation_filtered', 'dropped_by_correlation', 'mi_selected', 'rfe_selected', 'final_selected'])
Final selected count: 28
